## MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys, os
_stale = ['chromadb','gradio','sentence_transformers', 'pydantic',
          'huggingface_hub','langchain','transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# All project paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR,DATA_DIR,MDL_DIR,RES_DIR,RAG_DIR,CHR_DIR]:
    os.makedirs(d, exist_ok=True)

if LIB_DIR in sys.path: sys.path.remove(LIB_DIR)
sys.path.insert(0, LIB_DIR)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libs loaded | Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

## Load Fine-Tuned Model

In [ ]:
import torch, torch.nn as nn
import torchxrayvision as xrv

class FineTunedDenseNet(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        base = xrv.models.DenseNet(weights="densenet121-res224-all")
        self.features = base.features

        # This classifier includes the Pooling and Flattening layers internally
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.BatchNorm1d(1024),
            nn.Dropout(p=0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

        # This is the "Unexpected Key" from your error—it must be here!
        self.gradcam_layer = self.features.denseblock4

    def forward(self, x):
        if x.shape[1] == 3:
            x = x.mean(dim=1, keepdim=True)

        x = (x * 2048) - 1024
        feat = self.features(x)
        feat = torch.relu(feat)
        return self.classifier(feat)

In [ ]:
# Updated direct link from a different reliable GitHub repo
!wget -P {NIH_DIR} https://raw.githubusercontent.com/gregwchase/nih-chest-xray/master/data/BBox_List_2017.csv

## Grad-CAM++ IoU Evaluation

In [ ]:
# Grad-CAM++ evaluation against radiologist bounding boxes
# Computes IoU between model attention and ground truth annotations

import torch, json, os, cv2, numpy as np
import pandas as pd, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
import torchvision.transforms as T
from PIL import Image

DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))
NIH_DIR  = f"{DATA_DIR}/nih"

# Load fine-tuned model
ckpt  = torch.load(f"{MDL_DIR}/finetuned_densenet121_best.pth", map_location=DEVICE, weights_only=False)
model = FineTunedDenseNet(len(DISEASES)).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
target_layer = [model.gradcam_layer]

bbox_df = pd.read_csv(f"{NIH_DIR}/BBox_List_2017.csv")
print(f"Model loaded | {len(bbox_df)} bounding box annotations")

PREPROCESS = T.Compose([
    T.Resize((256,256)), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def find_image(name, base):
    p = os.path.join(base, name)
    if os.path.exists(p): return p
    import glob
    m = glob.glob(f"{base}/**/{name}", recursive=True)
    return m[0] if m else None

def bbox_to_mask(row, resize=224):
    """Convert NIH bounding box to binary mask resized to 224x224."""
    mask = np.zeros((1024, 1024), dtype=np.float32)
    # NIH CSV columns: Bbox [x, y, w, h]
    cols = bbox_df.columns.tolist()
    x = int(row[cols[2]]); y = int(row[cols[3]])
    w = int(row[cols[4]]); h = int(row[cols[5]])
    mask[y:y+h, x:x+w] = 1.0
    return (cv2.resize(mask, (resize, resize)) > 0).astype(np.float32)

def compute_iou(cam_map, gt_mask, thresh=0.4):
    cam_bin = (cam_map >= thresh).astype(np.float32)
    inter   = (cam_bin * gt_mask).sum()
    union   = np.clip(cam_bin + gt_mask, 0, 1).sum()
    return float(inter / (union + 1e-8))

# Run evaluation on all bounding box annotations
iou_results = []
sample_viz  = []
img_dir     = f"{NIH_DIR}/images"

with GradCAMPlusPlus(model=model, target_layers=target_layer) as cam:
    for _, row in tqdm(bbox_df.iterrows(), total=len(bbox_df), desc="Grad-CAM IoU"):
        img_file = row['Image Index']
        disease  = row['Finding Label']
        if disease not in DISEASES: continue
        img_path = find_image(img_file, img_dir)
        if not img_path: continue

        pil_img  = Image.open(img_path).convert('RGB')
        tensor   = PREPROCESS(pil_img).unsqueeze(0).to(DEVICE)
        class_idx = DISEASES.index(disease)

        cam_map  = cam(input_tensor=tensor,
                       targets=[ClassifierOutputTarget(class_idx)])[0]
        gt_mask  = bbox_to_mask(row)
        iou      = compute_iou(cam_map, gt_mask)
        iou_results.append({'image': img_file, 'disease': disease, 'iou': iou})

        if len(sample_viz) < 9:
            img_np  = np.array(pil_img.resize((224,224))).astype(np.float32)/255.0
            overlay = show_cam_on_image(img_np, cam_map, use_rgb=True)
            sample_viz.append((overlay, gt_mask, disease, iou))

# Results
iou_df = pd.DataFrame(iou_results)
iou_df.to_csv(f"{RES_DIR}/gradcam_iou_results.csv", index=False)
mean_iou = iou_df['iou'].mean()
per_d    = iou_df.groupby('disease')['iou'].mean().sort_values(ascending=False)

print(f"\n{'='*50}")
print(f"GRAD-CAM++ LOCALIZATION RESULTS")
print(f"{'='*50}")
print(f"Mean IoU : {mean_iou:.3f}")
for d, iou in per_d.items():
    bar = "█"*int(iou*30) + "░"*(30-int(iou*30))
    print(f"  {d:22} {bar} {iou:.3f}")

# Save mean_iou for app display
eval_res = json.load(open(f"{RES_DIR}/test_eval_results.json"))
eval_res['mean_iou'] = round(mean_iou, 3)
json.dump(eval_res, open(f"{RES_DIR}/test_eval_results.json", 'w'), indent=2)

# Visualize 9 examples with bounding boxes
fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.patch.set_facecolor('#08090c')
for ax, (cam_img, gt_mask, disease, iou) in zip(axes.flatten(), sample_viz):
    ax.imshow(cam_img)
    cont = np.zeros((*gt_mask.shape, 4), dtype=np.uint8)
    ctrs, _ = cv2.findContours((gt_mask*255).astype(np.uint8),
                                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(cont, ctrs, -1, (255,255,0,220), 2)
    ax.imshow(cont)
    c = '#22d3a0' if iou > 0.25 else '#f87171'
    ax.set_title(f"{disease}\nIoU={iou:.3f}", color=c, fontsize=9)
    ax.axis('off')
plt.suptitle(f"Grad-CAM++ vs Radiologist Annotations | Mean IoU={mean_iou:.3f}\nYellow=GT Box, Heatmap=Model Attention",
             color='white', fontsize=11)
plt.tight_layout()
plt.savefig(f"{RES_DIR}/gradcam_iou_examples.png", dpi=150,
            facecolor='#08090c', bbox_inches='tight')
plt.show()
print("Grad-CAM IoU evaluation complete!")